# Feedforward Neural Network

In [1]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

## Load the data

In [2]:
fp = "SPY_resistance.jsonl"

with open(fp, "r") as f:
    data = [json.loads(line) for line in f]

DATA = pd.DataFrame(data, columns=["hX", "lX", "hy", "ly"])

In [3]:
# Use the last 20 rows for manual testing
df = DATA.copy().iloc[:-20]

hX = df["hX"]
hy = df["hy"]
lX = df["lX"]
ly = df["ly"]

In [4]:
# Extract features and targets
hX = np.array(DATA["hX"].tolist()).astype(np.float32)
hy = np.array(DATA["hy"].tolist()).astype(np.float32)
lX = np.array(DATA["lX"].tolist()).astype(np.float32)
ly = np.array(DATA["ly"].tolist()).astype(np.float32)

# Split the data into train and test sets
train_hX, test_hX, train_hy, test_hy = train_test_split(hX, hy, test_size=0.2, random_state=42)
train_lX, test_lX, train_ly, test_ly = train_test_split(lX, ly, test_size=0.2, random_state=42)

## Train the model

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [6]:
class TimeSeriesDataset(Dataset):
    def __init__(self, features, targets):
        self.features = features
        self.targets = targets

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]


class TimeSeriesModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(TimeSeriesModel, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [7]:
# Training batch size
batch_size = 32

# Model architecture
input_size = 10
hidden_size = 64
output_size = 1

# Ensure the input size is correct
assert 10 == hX.shape[1]
assert 10 == lX.shape[1]

### Train the high model

In [8]:
# Create DataLoader instances
train_dataset_h = TimeSeriesDataset(train_hX, train_hy)
test_dataset_h = TimeSeriesDataset(test_hX, test_hy)

train_loader_h = DataLoader(train_dataset_h, batch_size=batch_size, shuffle=True)
test_loader_h = DataLoader(test_dataset_h, batch_size=batch_size, shuffle=False)

# Instantiate the model, loss function, and optimizer
model_h = TimeSeriesModel(input_size, hidden_size, output_size)
criterion_h = nn.MSELoss()
optimizer_h = torch.optim.Adam(model_h.parameters(), lr=0.001)

# Training loop
num_epochs = 20
for epoch in range(num_epochs):
    model_h.train()
    for batch_features, batch_targets in train_loader_h:
        # Forward pass
        outputs = model_h(batch_features)
        loss = criterion_h(outputs.squeeze(), batch_targets)

        # Backward pass and optimization
        optimizer_h.zero_grad()
        loss.backward()
        optimizer_h.step()

    print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}")

# Evaluate the model on the test set
model_h.eval()
total_loss = 0
with torch.no_grad():
    for batch_features, batch_targets in test_loader_h:
        outputs = model_h(batch_features)
        loss = criterion_h(outputs.squeeze(), batch_targets)
        total_loss += loss.item()

average_loss = total_loss / len(test_loader_h)
print(f"Average Test Loss: {average_loss:.4f}")

Epoch [1/20], Loss: 31000.7188
Epoch [2/20], Loss: 12.6022
Epoch [3/20], Loss: 98.1426
Epoch [4/20], Loss: 39.8070
Epoch [5/20], Loss: 16.1174
Epoch [6/20], Loss: 5.2229
Epoch [7/20], Loss: 19.8212
Epoch [8/20], Loss: 13.4101
Epoch [9/20], Loss: 3.9958
Epoch [10/20], Loss: 32.1634
Epoch [11/20], Loss: 14.9162
Epoch [12/20], Loss: 5.5757
Epoch [13/20], Loss: 163.1674
Epoch [14/20], Loss: 9.4133
Epoch [15/20], Loss: 7.4632
Epoch [16/20], Loss: 70.1631
Epoch [17/20], Loss: 24.5373
Epoch [18/20], Loss: 7.4899
Epoch [19/20], Loss: 12.7553
Epoch [20/20], Loss: 10.0034
Average Test Loss: 19.0601


### Train the low model

In [9]:
# Create DataLoader instances
train_dataset_l = TimeSeriesDataset(train_lX, train_ly)
test_dataset_l = TimeSeriesDataset(test_lX, test_ly)

train_loader_l = DataLoader(train_dataset_l, batch_size=batch_size, shuffle=True)
test_loader_l = DataLoader(test_dataset_l, batch_size=batch_size, shuffle=False)

# Instantiate the model, loss function, and optimizer
model_l = TimeSeriesModel(input_size, hidden_size, output_size)
criterion_l = nn.MSELoss()
optimizer_l = torch.optim.Adam(model_l.parameters(), lr=0.001)

# Training loop for the low model
num_epochs_l = 20
for epoch in range(num_epochs_l):
    model_l.train()
    for batch_features, batch_targets in train_loader_l:
        # Forward pass
        outputs = model_l(batch_features)
        loss = criterion_l(outputs.squeeze(), batch_targets)

        # Backward pass and optimization
        optimizer_l.zero_grad()
        loss.backward()
        optimizer_l.step()

    print(f"Epoch [{epoch + 1}/{num_epochs_l}], Loss: {loss.item():.4f}")

# Evaluate the model on the test set
model_l.eval()
total_loss_l = 0
with torch.no_grad():
    for batch_features, batch_targets in test_loader_l:
        outputs = model_l(batch_features)
        loss = criterion_l(outputs.squeeze(), batch_targets)
        total_loss_l += loss.item()

average_loss_l = total_loss_l / len(test_loader_l)
print(f"Average Test Loss for Low Model: {average_loss_l:.4f}")

Epoch [1/20], Loss: 5878.2241
Epoch [2/20], Loss: 1331.8873
Epoch [3/20], Loss: 69.7702
Epoch [4/20], Loss: 88.5463
Epoch [5/20], Loss: 34.8869
Epoch [6/20], Loss: 42.0852
Epoch [7/20], Loss: 8.3544
Epoch [8/20], Loss: 32.5904
Epoch [9/20], Loss: 43.7462
Epoch [10/20], Loss: 7.2399
Epoch [11/20], Loss: 15.8786
Epoch [12/20], Loss: 108.6087
Epoch [13/20], Loss: 31.4914
Epoch [14/20], Loss: 217.7972
Epoch [15/20], Loss: 27.3177
Epoch [16/20], Loss: 5.1215
Epoch [17/20], Loss: 32.0643
Epoch [18/20], Loss: 64.8719
Epoch [19/20], Loss: 37.6084
Epoch [20/20], Loss: 30.6850
Average Test Loss for Low Model: 26.3150


## Predict Weekly Resistance

In [10]:
from axiom.config import DATA_DIR

DAILY_DATA = pd.read_csv(DATA_DIR.joinpath("daily", "SPY_2024-07-20.csv"))

In [11]:
df = DAILY_DATA.copy()

# Filter columns
columns = ["datetime", "high", "low"]
df = df[columns]

# Convert int to datetime
df["datetime"] = pd.to_datetime(df["datetime"], unit="ms")

# Add a new column for the day of the week, 0=Monday, 6=Sunday
df["day"] = df["datetime"].dt.dayofweek
df.head()

,datetime,high,low,day
0,2004-07-19 05:00:00,110.96,109.99,0
1,2004-07-20 05:00:00,111.90,110.25,1
2,2004-07-21 05:00:00,112.06,109.45,2
3,2004-07-22 05:00:00,110.39,108.77,3
4,2004-07-23 05:00:00,109.71,108.69,4


In [12]:
# In the last 10 rows, get the index of the last "4" day
last_friday_index = df[df["day"] == 4].index[-1]

# Get the highs and lows of the last 5 days ending on the last "4" day
last_days = df.loc[last_friday_index - 9 : last_friday_index]

last_days_high = last_days["high"].values
last_days_low = last_days["low"].values
last_days_high, last_days_low

(array([556.2501, 557.18  , 561.67  , 562.33  , 563.67  , 564.8371,
        565.16  , 560.51  , 559.52  , 554.08  ]),
 array([554.19, 555.52, 556.77, 555.83, 557.15, 559.63, 562.1 , 556.61,
        550.43, 547.91]))

In [15]:
input_array = np.array(last_days_high, dtype=np.float32)
input_tensor = torch.tensor(input_array).unsqueeze(0)

# Set the model to evaluation mode
model_h.eval()
model_l.eval()

# Run inference
with torch.no_grad():
    prediction = model_h(input_tensor)
    predicted_high = prediction.item()

    prediction = model_l(input_tensor)
    predicted_low = prediction.item()

In [17]:
from datetime import timedelta

# Calculate the date of the next Monday
last_friday_date = df.iloc[last_friday_index]["datetime"]
next_monday_date = last_friday_date + timedelta(days=(7 - last_friday_date.weekday() + 1) % 7)

# Format the date to 'DD MMM YYYY'
formatted_date = next_monday_date.strftime("%d %b %Y")

# Print the formatted date with predicted high/low values
print(
    f"Prediction for week of {formatted_date}\n\tHigh: {predicted_high:.2f}\n\tLow: {predicted_low:.2f}"
)

Prediction for week of 23 Jul 2024
	High: 566.73
	Low: 556.12
